# Medical Diagnosis for Incomplete and Imbalanced Data
> **Published:** Springer, Intelligent Data Engineering and Analytics — Feb 28, 2022

This notebook demonstrates a full ML pipeline for disease risk prediction on healthcare datasets with:
- **Missing values** — handled via KNN Imputation
- **Class imbalance** — handled via SMOTE oversampling
- **Deep learning model** — multi-instance neural network (TensorFlow/Keras)

---

## 1. Install & Import Libraries

In [ ]:
# Uncomment to install
# !pip install tensorflow scikit-learn imbalanced-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print('All libraries loaded successfully!')

## 2. Load & Explore the Dataset

We simulate a realistic healthcare dataset with missing values and class imbalance — mirroring the conditions in the published paper.

> 💡 To use your own dataset: replace the cell below with `df = pd.read_csv('your_dataset.csv')`

In [ ]:
# ── Simulate a healthcare dataset ──────────────────────────────────────────
np.random.seed(42)
n_samples = 2000

X_raw, y_raw = make_classification(
    n_samples=n_samples,
    n_features=15,
    n_informative=10,
    n_redundant=3,
    weights=[0.85, 0.15],   # 85% healthy, 15% at-risk → imbalanced
    random_state=42
)

feature_names = [
    'age', 'bmi', 'blood_pressure', 'cholesterol', 'glucose',
    'insulin', 'heart_rate', 'hemoglobin', 'wbc_count', 'rbc_count',
    'platelet_count', 'sodium', 'potassium', 'creatinine', 'urea'
]

df = pd.DataFrame(X_raw, columns=feature_names)
df['disease_risk'] = y_raw

# Inject ~15% missing values to simulate real-world data
missing_mask = np.random.rand(*df.drop('disease_risk', axis=1).shape) < 0.15
df_missing = df.copy()
df_missing.loc[:, feature_names] = df_missing[feature_names].where(~missing_mask)

print(f'Dataset shape: {df_missing.shape}')
df_missing.head()

In [ ]:
# ── Basic statistics ────────────────────────────────────────────────────────
print('=== Dataset Info ===')
print(f'Total samples : {len(df_missing)}')
print(f'Features      : {len(feature_names)}')
print(f'\nClass distribution:')
print(df_missing['disease_risk'].value_counts().rename({0: 'Healthy', 1: 'At Risk'}))
print(f'\nImbalance ratio: {df_missing["disease_risk"].value_counts()[0] / df_missing["disease_risk"].value_counts()[1]:.1f}:1')

## 3. Exploratory Data Analysis

In [ ]:
# ── Missing value heatmap ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Missing values per feature
missing_pct = (df_missing[feature_names].isnull().sum() / len(df_missing) * 100).sort_values(ascending=False)
axes[0].bar(missing_pct.index, missing_pct.values, color='#E24B4A', alpha=0.8)
axes[0].set_title('Missing Values per Feature (%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Missing (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(y=15, color='gray', linestyle='--', alpha=0.5, label='Expected ~15%')
axes[0].legend()

# Class distribution
counts = df_missing['disease_risk'].value_counts()
axes[1].bar(['Healthy (0)', 'At Risk (1)'], counts.values, color=['#1D9E75', '#E24B4A'], alpha=0.8)
axes[1].set_title('Class Distribution (Imbalanced)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Sample Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature correlation heatmap ─────────────────────────────────────────────
plt.figure(figsize=(12, 8))
corr = df_missing[feature_names].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing

### 4a. KNN Imputation for Missing Values

In [ ]:
X = df_missing[feature_names].values
y = df_missing['disease_risk'].values

print(f'Missing values before imputation: {np.isnan(X).sum()}')

imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(X)

print(f'Missing values after imputation : {np.isnan(X_imputed).sum()}')
print('✅ KNN Imputation complete')

### 4b. SMOTE — Handling Class Imbalance

In [ ]:
print('Class distribution BEFORE SMOTE:')
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in zip(['Healthy', 'At Risk'], counts):
    print(f'  {cls}: {cnt} ({cnt/len(y)*100:.1f}%)')

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_imputed, y)

print('\nClass distribution AFTER SMOTE:')
unique, counts = np.unique(y_resampled, return_counts=True)
for cls, cnt in zip(['Healthy', 'At Risk'], counts):
    print(f'  {cls}: {cnt} ({cnt/len(y_resampled)*100:.1f}%)')
print('✅ SMOTE complete')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, labels, title in zip(axes,
    [y, y_resampled],
    ['Before SMOTE', 'After SMOTE']):
    unique, counts = np.unique(labels, return_counts=True)
    ax.bar(['Healthy', 'At Risk'], counts, color=['#1D9E75', '#E24B4A'], alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4c. Train/Test Split & Standardisation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

## 5. Build the Model

Multi-instance neural network with BatchNormalization and Dropout for regularisation.

In [ ]:
def build_model(input_dim):
    inputs = keras.Input(shape=(input_dim,), name='features')
    x = keras.layers.Dense(128, activation='relu')(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    outputs = keras.layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

## 6. Train the Model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=1),
    keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history.history[metric], label='Train', linewidth=2, color='#378ADD')
    ax.plot(history.history[f'val_{metric}'], label='Validation', linewidth=2,
            color='#E24B4A', linestyle='--')
    ax.set_title(f'Training {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation

In [ ]:
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Healthy', 'At Risk']))

roc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC Score: {roc:.4f}')

In [ ]:
# ── Confusion matrix + ROC curve ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Healthy', 'At Risk'],
            yticklabels=['Healthy', 'At Risk'])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], color='#378ADD')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_title(f'ROC Curve (AUC = {roc:.3f})', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key Findings

| Technique | Challenge Addressed | Result |
|---|---|---|
| KNN Imputation (k=5) | 15% missing clinical values | 0 missing values after imputation |
| SMOTE | 85:15 class imbalance | Balanced 50:50 training set |
| BatchNorm + Dropout | Overfitting | Stable train/val curves |
| Early Stopping | Unnecessary epochs | Best weights restored |

**Evaluation metrics used:** Precision, Recall, F1-score, and ROC-AUC — chosen specifically because accuracy alone is misleading on imbalanced healthcare datasets.

> *This work was published in Springer — Intelligent Data Engineering and Analytics, Feb 2022.*